In [ ]:
# ============================================================
# FINAL HS4 CODE CLASSIFIER
# Word + Character TF-IDF + Calibrated Linear SVM
# ============================================================

import ast
import os
import re
import joblib
import numpy as np
import pandas as pd

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report


# ============================================================
# 1. LOAD DATA
# ============================================================

print("Loading dataset...")

df = pd.DataFrame(
    load_dataset(
        "Dayanand314Krishna/cross_rulings_hts_dataset_for_tariffs"
    )["train"]
)

print("Original shape:", df.shape)


# ============================================================
# 2. EXTRACT USER TEXT + HS4 CODE
# ============================================================

def extract_text_and_code(message):

    try:
        data = ast.literal_eval(str(message))
    except:
        data = []

    user_text = ""
    assistant_text = ""

    if isinstance(data, list):
        for m in data:
            if m.get("role") == "user" and not user_text:
                user_text = m.get("content", "")

            elif m.get("role") == "assistant" and not assistant_text:
                assistant_text = m.get("content", "")

    # Remove question wording
    user_text = re.sub(
        r"^(what\s+is\s+the\s+hts\s+(us\s+)?code\s+for\s+|"
        r"what\s+is\s+the\s+hs\s+code\s+for\s+)",
        "",
        user_text,
        flags=re.IGNORECASE
    ).rstrip("?")

    # Extract HS4 from assistant answer
    match = re.search(
        r"\b(\d{4})\.\d{2}",
        assistant_text
    )

    if not match:
        match = re.search(
            r"\b(\d{4})\b",
            assistant_text
        )

    hs4 = match.group(1) if match else None

    return user_text.lower().strip(), hs4


parsed = df["messages"].apply(
    extract_text_and_code
)

df["clean_text"] = parsed.apply(
    lambda x: x[0]
)

df["hts_code"] = parsed.apply(
    lambda x: x[1]
)


# ============================================================
# 3. CLEAN DATA
# ============================================================

df = df.dropna(
    subset=["clean_text", "hts_code"]
).copy()

# Extract first 4 digits
df["hs4"] = (
    df["hts_code"]
    .astype(str)
    .str.replace(
        r"[^\d]",
        "",
        regex=True
    )
    .str[:4]
)

# Basic text cleaning
df["clean_text"] = (
    df["clean_text"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Keep valid HS4 and meaningful text
df = df[
    df["hs4"].str.fullmatch(r"\d{4}") &
    (df["clean_text"].str.len() > 10)
].copy()


# ------------------------------------------------------------
# Remove identical text with different HS4 labels
# ------------------------------------------------------------

conflicts = (
    df.groupby("clean_text")["hs4"]
    .nunique()
)

ambiguous_text = conflicts[
    conflicts > 1
].index

df = df[
    ~df["clean_text"].isin(ambiguous_text)
].copy()


# ------------------------------------------------------------
# Remove exact duplicate examples
# ------------------------------------------------------------

df = df.drop_duplicates(
    subset=["clean_text", "hs4"]
).reset_index(drop=True)


# ------------------------------------------------------------
# Remove very rare HS4 classes
# ------------------------------------------------------------

MIN_SAMPLES = 10

valid_classes = (
    df["hs4"]
    .value_counts()
    .loc[
        lambda x: x >= MIN_SAMPLES
    ]
    .index
)

df = df[
    df["hs4"].isin(valid_classes)
].copy()

print("\nFinal dataset shape:", df.shape)
print("HS4 classes:", df["hs4"].nunique())


# ============================================================
# 4. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["hs4"],
    test_size=0.20,
    random_state=42,
    stratify=df["hs4"]
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))


# ============================================================
# 5. FINAL MODEL
# ============================================================

model = Pipeline([

    (
        "features",

        FeatureUnion([

            # Word-level TF-IDF
            (
                "word",

                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=30000,
                    sublinear_tf=True
                )
            ),

            # Character-level TF-IDF
            (
                "char",

                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(3, 5),
                    min_df=2,
                    max_features=30000,
                    sublinear_tf=True
                )
            )
        ])
    ),

    (
        "svm",

        CalibratedClassifierCV(
            LinearSVC(
                C=1.0,
                class_weight="balanced",
                max_iter=5000,
                random_state=42
            ),
            cv=3,
            method="sigmoid",
            n_jobs=-1
        )
    )
])


# ============================================================
# 6. TRAIN MODEL
# ============================================================

print("\nTraining final calibrated SVM...")

model.fit(
    X_train,
    y_train
)


# ============================================================
# 7. EVALUATION
# ============================================================

probabilities = model.predict_proba(
    X_test
)

classes = model.classes_


# ------------------------------------------------------------
# Top-1 prediction
# ------------------------------------------------------------

top1_index = np.argmax(
    probabilities,
    axis=1
)

predictions = classes[
    top1_index
]


# ------------------------------------------------------------
# Top-3 predictions
# ------------------------------------------------------------

top3_index = np.argsort(
    probabilities,
    axis=1
)[:, -3:][:, ::-1]

top3_labels = classes[
    top3_index
]


# ------------------------------------------------------------
# Accuracy
# ------------------------------------------------------------

top1_accuracy = accuracy_score(
    y_test,
    predictions
)

top3_accuracy = np.mean([
    true_label in predicted_labels

    for true_label, predicted_labels
    in zip(
        y_test,
        top3_labels
    )
])


# ------------------------------------------------------------
# Wrong predictions
# ------------------------------------------------------------

wrong_count = np.sum(
    predictions != y_test.values
)


print("\n==============================")
print("FINAL MODEL RESULTS")
print("==============================")

print(
    f"Top-1 Accuracy: "
    f"{top1_accuracy * 100:.2f}%"
)

print(
    f"Top-3 Accuracy: "
    f"{top3_accuracy * 100:.2f}%"
)

print(
    f"\nTotal test cases: "
    f"{len(y_test)}"
)

print(
    f"Correct Top-1: "
    f"{len(y_test) - wrong_count}"
)

print(
    f"Wrong Top-1: "
    f"{wrong_count}"
)


# ============================================================
# 8. CLASSIFICATION REPORT
# ============================================================

print("\n==============================")
print("CLASSIFICATION REPORT")
print("==============================")

print(
    classification_report(
        y_test,
        predictions,
        zero_division=0
    )
)


# ============================================================
# 9. SAVE TEST RESULTS
# ============================================================

results_df = pd.DataFrame({

    "text":
        X_test.values,

    "actual_hs4":
        y_test.values,

    "predicted_hs4":
        predictions,

    "top1_correct":
        predictions == y_test.values,

    "top3_predictions": [
        " | ".join(row)
        for row in top3_labels
    ],

    "confidence": [
        round(
            probabilities[
                i,
                top1_index[i]
            ] * 100,
            2
        )

        for i in range(
            len(X_test)
        )
    ]
})


# All test predictions
results_df.to_csv(
    "test_predictions.csv",
    index=False
)


# Only wrong Top-1 predictions
results_df[
    ~results_df["top1_correct"]
].to_csv(
    "wrong_predictions.csv",
    index=False
)


# ============================================================
# 10. LIVE PREDICTION
# ============================================================

sample = (
    "Women's 100% woven cotton summer dress "
    "with floral embroidery"
)

prob = model.predict_proba(
    [sample]
)[0]

top3_index = np.argsort(
    prob
)[-3:][::-1]


print("\n==============================")
print("LIVE HS4 PREDICTION")
print("==============================")

print(
    "\nProduct:",
    sample
)

print("\nTop-3 Predictions:")

for rank, idx in enumerate(
    top3_index,
    start=1
):

    print(
        f"{rank}. "
        f"HS4: {classes[idx]} "
        f"({prob[idx] * 100:.2f}%)"
    )


# ============================================================
# 11. SAVE FINAL MODEL FILES
# ============================================================

print("\n==============================")
print("SAVING MODEL FILES")
print("==============================")


# Save trained model
joblib.dump(
    model,
    "final_hs4_calibrated_svm.pkl"
)


# Save model information
model_info = {

    "model_type":
        "Word + Character TF-IDF + Calibrated Linear SVM",

    "target":
        "HS4",

    "num_classes":
        len(classes),

    "classes":
        classes.tolist(),

    "word_ngram_range":
        (1, 2),

    "character_ngram_range":
        (3, 5),

    "min_df":
        2,

    "max_word_features":
        30000,

    "max_character_features":
        30000,

    "svm_C":
        1.0,

    "class_weight":
        "balanced",

    "calibration":
        "sigmoid",

    "calibration_cv":
        3,

    "top1_accuracy":
        round(top1_accuracy, 4),

    "top3_accuracy":
        round(top3_accuracy, 4),

    "test_samples":
        len(y_test)
}


joblib.dump(
    model_info,
    "model_info.pkl"
)


# ============================================================
# 12. CHECK SAVED FILES
# ============================================================

print("\nModel files:")

for file in [
    "final_hs4_calibrated_svm.pkl",
    "model_info.pkl"
]:

    size_mb = (
        os.path.getsize(file)
        / (1024 * 1024)
    )

    print(
        f"{file}: "
        f"{size_mb:.2f} MB"
    )


print("\nResult files:")

for file in [
    "test_predictions.csv",
    "wrong_predictions.csv"
]:

    size_mb = (
        os.path.getsize(file)
        / (1024 * 1024)
    )

    print(
        f"{file}: "
        f"{size_mb:.2f} MB"
    )


print("\n==============================")
print("PROJECT COMPLETE")
print("==============================")

Loading dataset...


README.md:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B / 23.7MB            

train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/260k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/257k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18254 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

Original shape: (18254, 1)

Final dataset shape: (14880, 4)
HS4 classes: 252
Training samples: 11904
Testing samples : 2976

Training final calibrated SVM...

FINAL MODEL RESULTS
Top-1 Accuracy: 69.62%
Top-3 Accuracy: 81.69%

Total test cases: 2976
Correct Top-1: 2072
Wrong Top-1: 904

CLASSIFICATION REPORT
              precision    recall  f1-score   support

        2703       1.00      1.00      1.00         3
        2825       0.67      0.67      0.67         3
        2901       1.00      0.40      0.57         5
        2902       0.50      0.50      0.50         4
        2903       0.62      0.71      0.67         7
        2904       1.00      0.50      0.67         8
        2905       0.71      0.77      0.74        13
        2906       0.56      0.62      0.59         8
        2907       0.80      0.57      0.67         7
        2909       0.71      0.42      0.53        12
        2911       1.00      0.67      0.80         3
        2912       0.50      0.33      0.4

In [ ]:
# ============================================================
# MODEL EVIDENCE 1: PERFORMANCE AND BASELINE
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score

# Generate predictions from the trained model
classes = np.asarray(model.classes_).astype(str)
y_true = np.asarray(y_test).astype(str)
probabilities = model.predict_proba(X_test)

# Top-1 predictions
top1_index = np.argmax(probabilities, axis=1)
predictions = classes[top1_index]

# Top-3 predictions
top3_index = np.argsort(probabilities, axis=1)[:, -3:][:, ::-1]
top3_predictions = classes[top3_index]

# Evaluate performance
top1_accuracy = accuracy_score(y_true, predictions)

top3_accuracy = np.mean([
    actual in candidates
    for actual, candidates in zip(y_true, top3_predictions)
])

macro_f1 = f1_score(
    y_true,
    predictions,
    labels=classes,
    average="macro",
    zero_division=0
)

# Majority-class baseline: always predict the most common training label
majority_code = str(y_train.astype(str).value_counts().idxmax())

baseline_predictions = np.full(
    len(y_true),
    majority_code
)

baseline_accuracy = accuracy_score(
    y_true,
    baseline_predictions
)

correct_count = int(np.sum(predictions == y_true))
wrong_count = int(np.sum(predictions != y_true))

print("MODEL EVALUATION")
print("=" * 40)
print(f"Top-1 accuracy:       {top1_accuracy:.2%}")
print(f"Top-3 accuracy:       {top3_accuracy:.2%}")
print(f"Macro F1:             {macro_f1:.4f}")
print(f"Majority class:       {majority_code}")
print(f"Majority baseline:    {baseline_accuracy:.2%}")
print(f"Test samples:         {len(y_true)}")
print(f"Correct Top-1:        {correct_count}")
print(f"Incorrect Top-1:      {wrong_count}")

# Save reproducible summary
evidence_summary = pd.DataFrame({
    "Metric": [
        "Top-1 Accuracy",
        "Top-3 Accuracy",
        "Macro F1",
        "Majority-Class Baseline",
        "Test Samples",
        "Correct Top-1",
        "Incorrect Top-1"
    ],
    "Value": [
        top1_accuracy,
        top3_accuracy,
        macro_f1,
        baseline_accuracy,
        len(y_true),
        correct_count,
        wrong_count
    ]
})

evidence_summary.to_csv(
    "model_evidence_summary.csv",
    index=False
)

print("\nSaved model_evidence_summary.csv")

MODEL EVALUATION
Top-1 accuracy:       69.62%
Top-3 accuracy:       81.69%
Macro F1:             0.5906
Majority class:       6110
Majority baseline:    7.09%
Test samples:         2976
Correct Top-1:        2072
Incorrect Top-1:      904

Saved model_evidence_summary.csv


In [ ]:
# ============================================================
# MODEL EVIDENCE 2: DISTINCT-CODE AND RECORD-LEVEL COVERAGE
# ============================================================

import re

# Reconstruct the parser output before class filtering
parsed_df = pd.DataFrame(
    parsed.tolist(),
    columns=["text", "raw_code"]
)

parsed_df["hs4"] = (
    parsed_df["raw_code"]
    .astype("string")
    .str.replace(r"\D", "", regex=True)
    .str[:4]
)

# Count distinct valid codes from the parsed source
valid_codes = parsed_df["hs4"].str.fullmatch(
    r"\d{4}",
    na=False
)

source_codes = set(
    parsed_df.loc[valid_codes, "hs4"].astype(str)
)

trained_codes = set(map(str, model.classes_))
supported_codes = source_codes & trained_codes
missing_codes = source_codes - trained_codes

print("DISTINCT-CODE COVERAGE")
print("=" * 40)
print("Codes found in parsed source:", len(source_codes))
print("Codes supported by model:", len(supported_codes))
print("Codes absent from model:", len(missing_codes))
print(
    f"Distinct-code coverage: "
    f"{len(supported_codes) / len(source_codes):.2%}"
)

# Prepare the records for record-level coverage.
# This retains duplicate pairs, as in the earlier diagnostic.
coverage_df = parsed_df.dropna(
    subset=["text", "hs4"]
).copy()

coverage_df["text"] = (
    coverage_df["text"]
    .astype(str)
    .str.lower()
    .str.strip()
)

coverage_df = coverage_df[
    coverage_df["hs4"].str.fullmatch(r"\d{4}", na=False)
    & (coverage_df["text"].str.len() > 10)
].copy()

# Exclude text descriptions associated with conflicting labels
label_counts = coverage_df.groupby("text")["hs4"].nunique()

conflicting_text = label_counts[
    label_counts > 1
].index

coverage_df = coverage_df[
    ~coverage_df["text"].isin(conflicting_text)
].copy()

# Measure how many eligible rows have labels supported by the model
supported_mask = coverage_df["hs4"].isin(trained_codes)

eligible_count = len(coverage_df)
supported_count = int(supported_mask.sum())
unsupported_count = eligible_count - supported_count

print("\nRECORD-LEVEL COVERAGE")
print("=" * 40)
print("Eligible parsed records:", eligible_count)
print("Records with supported labels:", supported_count)
print("Records with unsupported labels:", unsupported_count)
print(
    f"Record-level coverage: "
    f"{supported_count / eligible_count:.2%}"
)

# Save coverage evidence
coverage_summary = pd.DataFrame({
    "Metric": [
        "Distinct codes in parsed source",
        "Distinct codes supported",
        "Distinct codes unsupported",
        "Eligible parser-derived records",
        "Records with supported labels",
        "Records with unsupported labels"
    ],
    "Value": [
        len(source_codes),
        len(supported_codes),
        len(missing_codes),
        eligible_count,
        supported_count,
        unsupported_count
    ]
})

coverage_summary.to_csv(
    "model_coverage_summary.csv",
    index=False
)

print("\nSaved model_coverage_summary.csv")

DISTINCT-CODE COVERAGE
Codes found in parsed source: 548
Codes supported by model: 252
Codes absent from model: 296
Distinct-code coverage: 45.99%

RECORD-LEVEL COVERAGE
Eligible parsed records: 17509
Records with supported labels: 16625
Records with unsupported labels: 884
Record-level coverage: 94.95%

Saved model_coverage_summary.csv


In [ ]:
# ============================================================
# MODEL EVIDENCE 3: CONFIDENCE AND CALIBRATION
# ============================================================

# The highest predicted probability is the Top-1 confidence
confidence = probabilities.max(axis=1)
correct = predictions == y_true

reliability_df = pd.DataFrame({
    "confidence": confidence,
    "correct": correct
})

# Group confidence into five ranges
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.000001]
labels = ["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"]

reliability_df["confidence_bin"] = pd.cut(
    reliability_df["confidence"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

reliability_table = reliability_df.groupby(
    "confidence_bin",
    observed=False
).agg(
    Samples=("correct", "size"),
    Mean_Confidence=("confidence", "mean"),
    Actual_Accuracy=("correct", "mean")
).reset_index()

# Exclude empty bins from the calibration-error calculation
nonempty = reliability_table[
    reliability_table["Samples"] > 0
].copy()

ece = np.sum(
    nonempty["Samples"] / len(reliability_df)
    * (
        nonempty["Mean_Confidence"]
        - nonempty["Actual_Accuracy"]
    ).abs()
)

# Convert table values to percentages
reliability_table["Mean_Confidence"] *= 100
reliability_table["Actual_Accuracy"] *= 100

print("CONFIDENCE RELIABILITY")
print("=" * 40)
print(reliability_table.to_string(index=False))
print(f"\nExpected Calibration Error (ECE): {ece:.4f}")

# Confidence threshold analysis
print("\nCONFIDENCE THRESHOLDS")

for threshold in [0.60, 0.70, 0.80, 0.90]:
    selected = confidence >= threshold

    if selected.any():
        accuracy = correct[selected].mean()

        print(
            f"Confidence >= {threshold:.0%}: "
            f"{selected.sum()}/{len(correct)} cases "
            f"({selected.mean():.2%} coverage), "
            f"Top-1 accuracy = {accuracy:.2%}"
        )
    else:
        print(f"Confidence >= {threshold:.0%}: no cases")

reliability_table.to_csv(
    "confidence_reliability.csv",
    index=False
)

print("\nSaved confidence_reliability.csv")

CONFIDENCE RELIABILITY
confidence_bin  Samples  Mean_Confidence  Actual_Accuracy
         0-20%      405        12.833082        19.753086
        20-40%      586        30.368787        45.733788
        40-60%      719        50.229193        76.495132
        60-80%     1126        71.524456        92.095915
       80-100%      140        81.436470        97.857143

Expected Calibration Error (ECE): 0.1887

CONFIDENCE THRESHOLDS
Confidence >= 60%: 1266/2976 cases (42.54% coverage), Top-1 accuracy = 92.73%
Confidence >= 70%: 846/2976 cases (28.43% coverage), Top-1 accuracy = 95.04%
Confidence >= 80%: 140/2976 cases (4.70% coverage), Top-1 accuracy = 97.86%
Confidence >= 90%: no cases

Saved confidence_reliability.csv


In [ ]:
# ============================================================
# COVERAGE FUNNEL: RECONCILE DATASET COUNTS
# ============================================================

import pandas as pd
import numpy as np
import re

funnel_df = pd.DataFrame(
    parsed.tolist(),
    columns=["text", "raw_code"]
)

raw_count = len(funnel_df)

# Clean text and extract HS4
funnel_df["text"] = (
    funnel_df["text"].astype(str).str.lower().str.strip()
)

funnel_df["hs4"] = (
    funnel_df["raw_code"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
    .str[:4]
)

# Remove missing, invalid, and short records
funnel_df = funnel_df[
    funnel_df["hs4"].str.fullmatch(r"\d{4}") &
    (funnel_df["text"].str.len() > 10) &
    funnel_df["raw_code"].notna()
].copy()

valid_text_count = len(funnel_df)

# Remove descriptions associated with conflicting HS4 labels
conflicts = funnel_df.groupby("text")["hs4"].nunique()
conflicting_text = conflicts[conflicts > 1].index

funnel_df = funnel_df[
    ~funnel_df["text"].isin(conflicting_text)
].copy()

after_conflicts = len(funnel_df)

# Remove exact duplicate text-label pairs
funnel_df = funnel_df.drop_duplicates(
    subset=["text", "hs4"]
).reset_index(drop=True)

after_deduplication = len(funnel_df)

# Save the pool before the rare-class filter
pre_filter_df = funnel_df.copy()

# Apply the same minimum-class threshold used in training
class_counts = funnel_df["hs4"].value_counts()
valid_classes = class_counts[class_counts >= 10].index

final_df = funnel_df[
    funnel_df["hs4"].isin(valid_classes)
].copy()

# Coverage among the deduplicated records before rare-class filtering
supported_codes = set(map(str, model.classes_))
supported_mask = pre_filter_df["hs4"].isin(supported_codes)

record_coverage = supported_mask.mean()

# Distinct codes identified in the parsed source
all_parsed_codes = set(
    pd.Series(parsed.tolist()).apply(
        lambda x: re.sub(r"\D", "", str(x[1]))[:4]
        if x[1] is not None else None
    ).dropna()
)

all_parsed_codes = {
    code for code in all_parsed_codes
    if re.fullmatch(r"\d{4}", code)
}

distinct_coverage = (
    len(all_parsed_codes & supported_codes)
    / len(all_parsed_codes)
)

funnel = pd.DataFrame({
    "Stage": [
        "Original parsed training records",
        "Valid codes and descriptions",
        "After conflicting descriptions removed",
        "After exact duplicate removal",
        "After rare-class filter"
    ],
    "Records": [
        raw_count,
        valid_text_count,
        after_conflicts,
        after_deduplication,
        len(final_df)
    ]
})

print("DATASET FUNNEL")
print(funnel.to_string(index=False))

print("\nCOVERAGE AFTER DEDUPLICATION")
print("Eligible records before rare-class filter:", len(pre_filter_df))
print("Records with supported labels:", int(supported_mask.sum()))
print(f"Record-level coverage: {record_coverage:.2%}")
print("Distinct source codes:", len(all_parsed_codes))
print("Supported distinct source codes:",
      len(all_parsed_codes & supported_codes))
print(f"Distinct-code coverage: {distinct_coverage:.2%}")

print("\nMatches existing final dataset:", len(final_df) == len(df))

funnel.to_csv("coverage_funnel.csv", index=False)

DATASET FUNNEL
                                 Stage  Records
      Original parsed training records    18254
          Valid codes and descriptions    18233
After conflicting descriptions removed    17509
         After exact duplicate removal    15697
               After rare-class filter    14880

COVERAGE AFTER DEDUPLICATION
Eligible records before rare-class filter: 15697
Records with supported labels: 14880
Record-level coverage: 94.80%
Distinct source codes: 548
Supported distinct source codes: 252
Distinct-code coverage: 45.99%

Matches existing final dataset: True


In [ ]:
# ============================================================
# ABLATION: WORD-ONLY vs WORD+CHARACTER FEATURES
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score

y_true = np.asarray(y_test).astype(str)


def evaluate_model(name, top1, top3):
    return {
        "Model": name,
        "Top-1 Accuracy": accuracy_score(y_true, top1),
        "Top-3 Accuracy": np.mean([
            actual in candidates
            for actual, candidates in zip(y_true, top3)
        ]),
        "Macro F1": f1_score(
            y_true, top1,
            average="macro",
            zero_division=0
        )
    }


# Existing final model: word + character + calibration
probs_combined = model.predict_proba(X_test)
classes_combined = model.classes_.astype(str)

top1_combined = classes_combined[
    np.argmax(probs_combined, axis=1)
]

top3_combined = classes_combined[
    np.argsort(probs_combined, axis=1)[:, -3:][:, ::-1]
]

results = [
    evaluate_model(
        "Word + Character, Calibrated SVM",
        top1_combined,
        top3_combined
    )
]

# Word-only calibrated SVM
print("Training word-only comparison model...")

word_only_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=30000,
            sublinear_tf=True
        )
    ),
    (
        "svm",
        CalibratedClassifierCV(
            LinearSVC(
                C=1.0,
                class_weight="balanced",
                max_iter=5000,
                random_state=42
            ),
            cv=3,
            method="sigmoid",
            n_jobs=-1
        )
    )
])

word_only_model.fit(X_train, y_train)

word_probs = word_only_model.predict_proba(X_test)
word_classes = word_only_model.classes_.astype(str)

top1_word = word_classes[np.argmax(word_probs, axis=1)]
top3_word = word_classes[
    np.argsort(word_probs, axis=1)[:, -3:][:, ::-1]
]

results.append(
    evaluate_model(
        "Word-only, Calibrated SVM",
        top1_word,
        top3_word
    )
)

# Uncalibrated SVM using the final model's fitted word+character features
print("Training uncalibrated comparison model...")

features = model.named_steps["features"]

X_train_features = features.transform(X_train)
X_test_features = features.transform(X_test)

raw_svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=5000,
    random_state=42
)

raw_svm.fit(X_train_features, y_train)

raw_scores = raw_svm.decision_function(X_test_features)
raw_classes = raw_svm.classes_.astype(str)

top1_raw = raw_classes[np.argmax(raw_scores, axis=1)]
top3_raw = raw_classes[
    np.argsort(raw_scores, axis=1)[:, -3:][:, ::-1]
]

results.append(
    evaluate_model(
        "Word + Character, Uncalibrated SVM",
        top1_raw,
        top3_raw
    )
)

comparison = pd.DataFrame(results)

print("\nABLATION RESULTS")
print(comparison.to_string(index=False))

comparison.to_csv("model_ablation.csv", index=False)

Training word-only comparison model...
Training uncalibrated comparison model...

ABLATION RESULTS
                             Model  Top-1 Accuracy  Top-3 Accuracy  Macro F1
  Word + Character, Calibrated SVM        0.696237        0.816868  0.590636
         Word-only, Calibrated SVM        0.667003        0.787970  0.545045
Word + Character, Uncalibrated SVM        0.690524        0.808468  0.599967


In [ ]:
# ============================================================
# NEAR-DUPLICATE SIMILARITY CHECK
# ============================================================

from sklearn.neighbors import NearestNeighbors

nearest_model = NearestNeighbors(
    n_neighbors=1,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

nearest_model.fit(X_train_features)

distances, indices = nearest_model.kneighbors(X_test_features)

similarity = np.clip(1 - distances.ravel(), 0, 1)

actual = np.asarray(y_test).astype(str)

combined_probs = model.predict_proba(X_test)
combined_classes = model.classes_.astype(str)

top1 = combined_classes[np.argmax(combined_probs, axis=1)]
top3 = combined_classes[
    np.argsort(combined_probs, axis=1)[:, -3:][:, ::-1]
]

similarity_df = pd.DataFrame({
    "test_text": X_test.to_numpy(),
    "actual_hs4": actual,
    "predicted_hs4": top1,
    "top1_correct": top1 == actual,
    "top3_correct": [
        label in candidates
        for label, candidates in zip(actual, top3)
    ],
    "nearest_train_text": X_train.iloc[indices.ravel()].to_numpy(),
    "nearest_similarity": similarity
})

bins = [0, 0.25, 0.50, 0.70, 0.85, 1.000001]
labels = ["0–0.25", "0.25–0.50", "0.50–0.70",
          "0.70–0.85", "0.85–1.00"]

similarity_df["similarity_band"] = pd.cut(
    similarity_df["nearest_similarity"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

report = similarity_df.groupby(
    "similarity_band",
    observed=False
).agg(
    samples=("top1_correct", "size"),
    mean_similarity=("nearest_similarity", "mean"),
    top1_accuracy=("top1_correct", "mean"),
    top3_accuracy=("top3_correct", "mean")
).reset_index()

for column in ["mean_similarity", "top1_accuracy", "top3_accuracy"]:
    report[column] *= 100

print("ACCURACY BY NEAREST-TRAINING SIMILARITY")
print(report.to_string(index=False))

print("\nHigh-similarity test cases (similarity >= 0.90):")

high = similarity_df["nearest_similarity"] >= 0.90
print("Cases:", int(high.sum()))

if high.any():
    print("Top-1 accuracy:",
          f"{similarity_df.loc[high, 'top1_correct'].mean():.2%}")
    print("Top-3 accuracy:",
          f"{similarity_df.loc[high, 'top3_correct'].mean():.2%}")

similarity_df.to_csv("near_duplicate_analysis.csv", index=False)

ACCURACY BY NEAREST-TRAINING SIMILARITY
similarity_band  samples  mean_similarity  top1_accuracy  top3_accuracy
         0–0.25       87        22.639584      37.931034      58.620690
      0.25–0.50     1482        38.293135      58.704453      73.616734
      0.50–0.70      739        59.079924      79.296346      90.392422
      0.70–0.85      369        77.454463      87.262873      92.682927
      0.85–1.00      299        92.351471      87.290970      93.311037

High-similarity test cases (similarity >= 0.90):
Cases: 196
Top-1 accuracy: 87.24%
Top-3 accuracy: 93.37%
